In [1]:
import os.path
import traceback

from util.download_util import download_files
from util.dta_file_utils import find_dta_file
import asyncio
import pandas as pd
import zipfile

In [2]:
STATA_BASE_URL = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NHAMCS/stata/"
DATA_BASE_PATH = "data"
os.makedirs(DATA_BASE_PATH,exist_ok=True)

In [3]:
urls = []
years = ["2019", "2020", "2021","2022"]
file_names=[]
for year in years:
    file_name = f"ED{year}-stata.zip"
    url = STATA_BASE_URL + file_name
    urls.append(url)
    file_names.append(os.path.join(DATA_BASE_PATH,file_name))

In [4]:
await download_files(urls, file_names)

data\ED2022-stata.zip: 100%|██████████| 2.31M/2.31M [00:00<00:00, 2.57MB/s]
data\ED2019-stata.zip: 100%|██████████| 2.74M/2.74M [00:00<00:00, 2.89MB/s]
data\ED2020-stata.zip: 100%|██████████| 2.12M/2.12M [00:01<00:00, 1.68MB/s]
data\ED2021-stata.zip: 100%|██████████| 4.64M/4.64M [00:01<00:00, 2.79MB/s]


In [5]:
dataframes = dict()
for file_name in file_names:
    with zipfile.ZipFile(file_name, "r") as z:
        # Open the .dta file inside the ZIP
        for stata_file in find_dta_file(z.namelist()):
            print(stata_file)
            with z.open(stata_file) as f:
                try:
                    dataframes[stata_file] = pd.read_stata(f,convert_categoricals=False)
                except Exception as e:
                    print("Could not read stata file:", stata_file)
                    traceback.print_exc()



ED2019-stata.dta
ed2020-stata.dta
ed2021-stata.dta
ed2022-stata.dta


In [6]:
dataframes

{'ED2019-stata.dta':        VMONTH  VDAYR ARRTIME  WAITTIME  LOV  AGE  AGER  AGEDAYS  RESIDNCE  \
 0           3      4    1049         3  138   53     4       -7         3   
 1           3      2    2109         3  133    2     1       -7         1   
 2           3      6    1543         2  211    2     1       -7         1   
 3           3      6    0152         3  151   21     2       -7         3   
 4           3      7    0325         5  766   31     3       -7         3   
 ...       ...    ...     ...       ...  ...  ...   ...      ...       ...   
 19476       4      6    2156       169  254   26     3       -7         1   
 19477       4      3    0936        30   87   11     1       -7         1   
 19478       4      6    1142       163  180   94     6       -7         1   
 19479       5      5    1434        33  313   67     5       -7         1   
 19480       5      4    0913        31  121   28     3       -7         1   
 
        SEX  ...  RX30V3C2  RX30V3C3  RX30